In [2]:
import pandas as pd
import numpy as np
import os

print("INTEGRATING 4 POINT DATASETS")
print("="*80)

data_dir = "/scratch.global/ocon0444/peat_modeling/00_data/point_data"

df1 = pd.read_csv(f"{data_dir}/peat_depths.csv")
print(f"Original peat depths: {len(df1)} points")

df2 = pd.read_csv(f"{data_dir}/NASIS_known_depth.csv")
print(f"NASIS known depth: {len(df2)} points")

df3 = pd.read_csv(f"{data_dir}/NASIS_fully_organic.csv")
print(f"NASIS fully organic: {len(df3)} points")

df4 = pd.read_excel(f"{data_dir}/releve_site_export_edited.xlsx")
print(f"Releve sites: {len(df4)} points")

print("\nStandardizing datasets...")

std1 = pd.DataFrame({
    'source': 'original',
    'lat': np.nan,
    'lon': np.nan,
    'utm_e': df1['UTM_E83'],
    'utm_n': df1['UTM_N83'],
    'depth_cm': df1['depb'],
    'peat_binary': (df1['depb'] >= 20).astype(int),
    'use_for_depth': True,
    'siteid': df1['siteid']
})

std2 = pd.DataFrame({
    'source': 'nasis_known',
    'lat': df2['latstddecimaldegrees'],
    'lon': df2['longstddecimaldegrees'],
    'utm_e': np.nan,
    'utm_n': np.nan,
    'depth_cm': df2['o_thickness_cm'],
    'peat_binary': (df2['o_thickness_cm'] >= 20).astype(int),
    'use_for_depth': True,
    'siteid': df2['peiid'].astype(str)
})

std3 = pd.DataFrame({
    'source': 'nasis_fully_organic',
    'lat': df3['latstddecimaldegrees'],
    'lon': df3['longstddecimaldegrees'],
    'utm_e': np.nan,
    'utm_n': np.nan,
    'depth_cm': df3['o_thickness_cm'],
    'peat_binary': 1,
    'use_for_depth': False,
    'siteid': df3['peiid'].astype(str)
})

std4 = pd.DataFrame({
    'source': 'releve',
    'lat': np.nan,
    'lon': np.nan,
    'utm_e': df4['utm_easting'],
    'utm_n': df4['utm_northing'],
    'depth_cm': np.nan,
    'peat_binary': (df4['value'] > 0).astype(int),
    'use_for_depth': False,
    'siteid': df4['id'].astype(str)
})

merged = pd.concat([std1, std2, std3, std4], ignore_index=True)

merged = merged.dropna(subset=['utm_e', 'utm_n', 'lat', 'lon'], how='all')

print("\n" + "="*80)
print("INTEGRATION SUMMARY")
print("="*80)
print(f"Total points: {len(merged)}")
print(f"For binary classification: {len(merged)}")
print(f"For depth classification: {merged['use_for_depth'].sum()}")
print(f"\nPeat present: {merged['peat_binary'].sum()} ({100*merged['peat_binary'].mean():.1f}%)")
print(f"No peat: {(merged['peat_binary']==0).sum()} ({100*(1-merged['peat_binary'].mean()):.1f}%)")

print(f"\nBy source:")
for source in merged['source'].unique():
    subset = merged[merged['source']==source]
    print(f"  {source}: {len(subset)} points")

out_file = f"{data_dir}/integrated_points.csv"
merged.to_csv(out_file, index=False)
print(f"\nSaved: {out_file}")

INTEGRATING 4 POINT DATASETS
Original peat depths: 13079 points
NASIS known depth: 10708 points
NASIS fully organic: 1585 points
Releve sites: 11493 points

Standardizing datasets...

INTEGRATION SUMMARY
Total points: 36865
For binary classification: 36865
For depth classification: 23787

Peat present: 15026 (40.8%)
No peat: 21839 (59.2%)

By source:
  original: 13079 points
  nasis_known: 10708 points
  nasis_fully_organic: 1585 points
  releve: 11493 points

Saved: /scratch.global/ocon0444/peat_modeling/00_data/point_data/integrated_points.csv
